In [4]:
%pip install tldextract

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

phishing = pd.read_csv('Phising_DS.csv', header=None)
legit = pd.read_csv('Legitimate_DS.csv', header=None)

phishing.columns = ['url']
legit.columns = ['url']

phishing['label'] = 1
legit['label'] = 0

df = pd.concat([phishing, legit], ignore_index=True)

print(df.head())

df['url'] = df['url'].str.lower().str.strip()

# Hapus NaN
df = df.dropna(subset=['url'])

# Convert ke string (penting!)
df['url'] = df['url'].astype(str)

# Hapus yang terlalu aneh
df = df[df['url'].str.len() > 3]
df = df[~df['url'].isin(['.', '-', 'nan'])]

from urllib.parse import urlparse

def safe_url_parse(url):
    try:
        url = str(url).strip().lower()
        
        # Skip invalid
        if url in ['', '.', '-', 'nan']:
            return None, None, None
        
        # Tambahkan scheme kalau tidak ada
        if not url.startswith(('http://', 'https://')):
            url = 'http://' + url
        
        parsed = urlparse(url)
        
        return parsed.netloc, parsed.path, parsed.scheme
    
    except:
        return None, None, None

parsed_data = df['url'].apply(safe_url_parse)

df[['domain', 'path', 'scheme']] = pd.DataFrame(parsed_data.tolist(), index=df.index)

df = df.dropna(subset=['domain'])

# Panjang URL
df['url_length'] = df['url'].apply(len)

# Jumlah titik (.)
df['dot_count'] = df['url'].apply(lambda x: x.count('.'))

# Ada HTTPS atau tidak
df['has_https'] = df['scheme'].fillna('').apply(lambda x: 1 if x == 'https' else 0)

# Jumlah subdomain
df['subdomain_count'] = df['domain'].apply(lambda x: x.count('.') - 1)

# Ada IP di URL (indikator phishing kuat)
import re
df['has_ip'] = df['url'].apply(lambda x: 1 if re.search(r'(\d{1,3}\.){3}\d{1,3}', x) else 0)

# Ada simbol mencurigakan
df['has_at'] = df['url'].apply(lambda x: 1 if '@' in x else 0)
df['has_dash'] = df['url'].apply(lambda x: 1 if '-' in x else 0)

import tldextract

def get_root_domain(url):
    try:
        ext = tldextract.extract(url)
        return ext.domain + '.' + ext.suffix
    except:
        return None

df['root_domain'] = df['url'].apply(get_root_domain)


df['domain'].dropna().to_csv('domains.txt', index=False, header=False)

                                                 url  label
0                   http://atualizacaodedados.online      1
1                     http://webmasteradmin.ukit.me/      1
2           http://stcdxmt.bigperl.in/klxtv/apps/uk/      1
3   https://tubuh-syarikat.com/plugins/fields/files/      1
4  http://rolyborgesmd.com/exceword/excel.php?.ra...      1


In [ ]:
import socket

socket.setdefaulttimeout(2)

def get_ip(domain):
    try:
        return socket.gethostbyname(domain)
    except:
        return None

df['ip_address'] = df['domain'].apply(get_ip)

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

numeric_cols = ['url_length', 'dot_count']

df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [ ]:
df.to_csv('clean_dataset.csv', index=False)